# Testing Qwen

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install qwen_vl_utils

from transformers import AutoModelForCausalLM, AutoProcessor, AutoModelForVision2Seq
from qwen_vl_utils import process_vision_info
import torch
from PIL import Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 63.8 MB/s eta 0:00:00


### Load in Data

In [ ]:
### LOAD IN DATA ###
import pandas as pd

demo_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/demo_set.csv"
demo_df = pd.read_csv(demo_data_path)

test_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/testing_set.csv"
testing_df = pd.read_csv(test_data_path)

In [ ]:
testing_df.head()

,art_style,painting,emotion,utterance,repetition,embeddings
0,Abstract_Expressionism,hans-hofmann_nulli-secundus-1964,contentment,The painting makes me feel like I am walking d...,5,"[-0.0017447734717279673, -0.04706191271543503,..."
1,Color_Field_Painting,gene-davis_color-needles-1984,contentment,Lines are arranged in a perfect order.,5,"[-0.02143058553338051, 0.036608025431632996, 0..."
2,Cubism,arshile-gorky_bound-in-sleep,amusement,"The nose and eyes are too big for this face, m...",5,"[-0.03630712255835533, 0.002491589169949293, -..."
3,Minimalism,pino-pinelli_pittura-b-g-1991,amusement,The fruity donuts glued onto the wall is funny...,5,"[-0.05934969708323479, 0.024623168632388115, 0..."
4,Cubism,pyotr-konchalovsky_still-life-1911,something else,i dont feel anything when I look at this,5,"[-0.005745342001318932, 0.034919410943984985, ..."


### Load in Model

In [ ]:
model_id = "Qwen/Qwen3-VL-8B-Thinking"

# Load model + processor
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
# Define start of prompt
start_of_prompt = '''
STEP 1: How does this painting make you primarily feel? Select from the following: Amusement, Awe, Contentment, Excitement, Fear, Sadness, Anger, Something-Else

then...

STEP 2: Give a detailed description (at least 7 words) about WHY you feel like this, based on SPECIFIC details of the painting.

Examples of GOOD descriptions:
  - "the sky looks gloomy and the shadows are scary"
  - "the red marks on the table look like drops of blood" (analogies are good!)
  - "the blue color of the lake contrasts with the orange hats of the men"

Examples of BAD descriptions:
  - Vague descriptions that do not explain WHY you felt like this:
    - "I like its colors" (be more specific)
    - "It is amazing, nice work! great painting" (why is it amazing? be specific)
    - "I don't know what this is" (try and explain what it looks like)

Note: if you selected "Something-Else" also explain how you felt in addition to why you felt Something-Else.

Finally, **output your response strictly in the following JSON format**:
{
    "emotion": string,
    "utterance": string
}

Please make sure that once you have fully thought through your answers and you respond with them in the above JSON Format and only output that JSON.
Also only output the JSON for the emotion that best describes how the painting makes you feel.
'''

In [ ]:
### Select Random Images For Demonstrations ###
def get_random_images(df, n):
    # Sample n rows at once
    random_rows = df.sample(n=n, replace=False)

    demos = []

    for i in range(n):
        row = random_rows.iloc[i]

        art_style = row["art_style"].lower()
        painting = row["painting"].lower()

        image_path = (
            f"/content/drive/Shareddrives/LLMs_Art_Project/Data/"
            f"{art_style}_images/{painting}.jpeg"
        )

        demos.append((image_path, row))

    return demos

In [ ]:
import json

def parse_model_output(output_text):
    try:
        # Extract only the JSON portion the model produced
        json_start = output_text.find("{")
        json_end = output_text.rfind("}") + 1
        json_str = output_text[json_start:json_end]
        data = json.loads(json_str)

        emotion = data.get("emotion", "")
        utterance = data.get("utterance", "")
        return emotion, utterance

    except Exception as e:
        print("JSON parse error:", e)
        return "", ""  # fallback if model output isn't valid JSON

In [ ]:
def run_qwen(messages, max_new_tokens=256):
    """
    Run inference on Qwen using a pre-formatted messages list.
    `messages` must be a list of dicts in the specified chat format.
    """
    chat_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, format="json"
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=2000,
        temperature=0.2,
    )

    generated = output[0][len(inputs.input_ids[0]):]
    caption = processor.decode(generated, skip_special_tokens=False)

    return caption


In [ ]:
from numpy import number
# Loop through testing dataframe (aka run one inference on each testing image)
for number_demos in [0, 1, 2, 3 4]:
    results = []

    for index, row in testing_df.iterrows():
        # Extract test sample info
        test_art_style = row["art_style"].lower()
        test_painting_name = row["painting"].lower()
        test_emotion_gt = row["emotion"]
        test_utterance_gt = row["utterance"]

        # Build full file path for the test image
        test_image_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/{test_art_style}_images/{test_painting_name}.jpeg"
        # print(f"Processing test image {test_painting_name}")

        # Build input message
        content = []
        # Start of prompt
        content.append({"type": "text", "text": start_of_prompt})

        demo_painting_names = []
        demo_utterances = []

        # Add demos into prompt
        if number_demos > 0:
            # Set up demos
            content.append({"type": "text", "text": f"Here are {number_demos} examples of how to respond:"})

            # Get number_demos random (image_path, row) pairs
            demo_samples = get_random_images(demo_df, number_demos)

            # Unpack demo samples
            for i, (demo_image_path, demo_row) in enumerate(demo_samples, start=1):
                demo_emotion = demo_row["emotion"]
                demo_utterance = demo_row["utterance"]
                demo_painting = demo_row["painting"].lower()

                # Store for later
                demo_painting_names.append(demo_painting)
                demo_utterances.append(demo_utterance)

                # Add to prompt
                content.append({"type": "text", "text": f"Example {i}:"})
                content.append({"type": "image", "image": demo_image_path})
                content.append({"type": "text", "text": f"Emotion: {demo_emotion}\nReason: {demo_utterance}"})


        # Now add the actual test image
        content.append({"type": "text", "text": "Now analyze this painting."})
        content.append({"type": "image", "image": test_image_path})

        # Build the message
        messages = [
          {
              "role": "user",
              "content": content,
          }
        ]


        #print("Input")
        #print(messages)
        # Run the generation, note: switch function name per model
        print("Generating...")
        generated_output = run_qwen(messages)

        # Parse the JSON (predicted emotion + utterance)
        pred_emotion, pred_utterance = parse_model_output(generated_output)

        # STORE RESULTS
        # ------------------------
        results.append({
            "test_painting": test_painting_name,
            "demo_paintings": demo_painting_names,
            "demo_utterances": demo_utterances,
            "predicted_emotion": pred_emotion,
            "generated_description": pred_utterance
        })

        # Print out generated output
        print(f'Generated output for test image {index} titled {test_painting_name}:')
        print(f'{generated_output}')
        print(f'Predicted emotion: {pred_emotion}')
        print(f'Predicted utterance: {pred_utterance}')
        print('')

    # Store results in a dataframe
    results_df = pd.DataFrame(results)
    # Build Output Path
    output_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Qwen/qwen_results_{number_demos}shot.csv"
    # Save to csv file
    results_df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")



Streaming output truncated to the last 5000 lines.

Generating...
Generated output for test image 18 titled jackson-pollock_number-12-1949:
Got it, let's tackle this. First, I need to figure out the primary emotion from the options. The painting is an abstract expressionist piece with lots of chaotic lines, splatters, and mixed colors. Let's think: the energy is high, the lines are frantic, so maybe excitement or awe? Wait, the description says "chaotic" but also vibrant. Let's check the examples. The bad examples are vague, so I need specific details.

Looking at the painting: it's full of energetic, overlapping lines in various colors—brown, yellow, black, white, red. The brushstrokes are wild, almost like a storm. So maybe excitement? Or awe? Let's see. Awe is more about wonder, but the chaotic energy might feel exciting. Wait, the user's examples: "the sky looks gloomy and the shadows are scary" is fear. So for this, the chaotic lines and vibrant colors might evoke excitement. Let'

## EXTRA CODE

In [ ]:
### BUILD MESSAGE TO PASS INTO FUNCTION ###

# Get image path and image descriptions for demonstrations
# Currently using 0 shot learning so no demonstrations needed
n = 2
demo_image_1 = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_chicago-6-1961.jpeg"
demo_description_1 = "The black sharp lines running throughout the composition is jarring and uneasy."
demo_image_2 = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_new-york-2-1948.jpeg"
demo_description_2 = "Jagged sharp edges of shapes make me feel fear"



# Get image path for testing image
test_image_path = "/content/drive/Shareddrives/LLMs_Art_Project/Data/abstract_expressionism_images/aaron-siskind_acolman-1-1955.jpeg"




# Prepare message — image + text prompt
messages = [
    {
            "role": "user",
            "content": [
                {"type": "text", "text": text_prompt},
                {"type": "text", "text": f"Here are {n} examples of how to respond:"},
                {"type": "text", "text": "Example 1:"},
                {"type": "image", "image": demo_image_1},
                {"type": "text", "text": demo_description_1},
                {"type": "text", "text": "Example 2:"},
                {"type": "image", "image": demo_image_2},
                {"type": "text", "text": demo_description_2},
                {"type": "text", "text": "This is the art I want you to analyze"},
                {"type": "image", "image": test_image_path},
            ],
        }
    ]

In [ ]:
generated_output = run_qwen(messages)

In [ ]:
image = Image.open(test_image_path).convert("RGB")
display(image)
print("{" + generated_output.split("{")[1].split("}")[0] + "}")

In [ ]:
### EXAMPLE ###
'''
content = [
    {"type": "text", "text": text_prompt},
    {"type": "text", "text": f"First, however, I am going to show you {n} examples"},
    {"type": "text", "text": "Example 1:"},
    {"type": "image", "image": image_path1},
    {"type": "text", "text": "Example 2:"},
    {"type": "image", "image": image_path1},
    ...,
    {"type": "text", "text": "This is the art I want you to analyze"},
    {"type": "image", "image": image_path}
]
'''